# **Sprint 2 - Prompt e Inteligência Artificial**

**Integrantes:**

*   Carlos Eduardo Affonso — RM 569676
*   Gabriel Oliveira Gusmão Florencio dos Santos — RM 573747
*   Gabrieli de Lima Pettena de Oliveira — RM 569799
*   Igor Massone Monteiro — RM 573853
*   Temitope Kuku da Silva Ogunbanjo — RM 573772

In [ ]:
%pip install --quiet --upgrade langchain langchain-core langchain-community langchain-openai langgraph langchain-text-splitters pypdf unstructured pydantic

In [2]:
# configurando chatgpt
import getpass
import os
from google.colab import userdata
from langchain.chat_models import init_chat_model
from langchain_core.vectorstores import InMemoryVectorStore

In [3]:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
llm = init_chat_model("gpt-4o-mini", model_provider="openai")

In [4]:
#selecionando o embedding
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

In [5]:
vector_store = InMemoryVectorStore(embeddings)

In [ ]:
#criando uma base de cohecimento
import bs4
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader

file_path = "/content/GW_HCA-G2_User-Manual-PT.pdf"
loader1 = PyPDFLoader(file_path)
loader2 = WebBaseLoader(
    ['https://www.repsol.pt/particulares/assessoramento/carregar-carros-eletricos-chuva/',
     'https://www.enelx.com/br/pt/conteudos/guia-carregamento-carro-eletrico',
     'https://www.neocharge.com.br/tudo-sobre/carro-eletrico/tempo-carga-veiculo-eletrico?srsltid=AfmBOopTW2cw848TEwGUtiYjiztiCVs1NWyYqtkq4KbrTQ_X96VVyV-L',
     'https://www.neocharge.com.br/tudo-sobre/carregador-rapido-dc',
     'https://www.greenv.com.br/blog/em-quanto-tempo-carrega-meu-carro-eletrico/',
     'https://www.gov.br/inmetro/pt-br/assuntos/avaliacao-da-conformidade/programa-brasileiro-de-etiquetagem/tabelas-de-eficiencia-energetica/veiculos-automotivos-pbe-veicular?utm_source=copilot.com',
     'https://www.greenv.com.br/blog/quanto-tempo-demora-para-carregar-carro-eletrico-em-eletroposto/'
      ]

)
docs1 = loader1.load()
docs2 = loader2.load()
docs = docs1 + docs2

In [7]:
#fazendo o splitting dos documentos
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

In [8]:
#guardando os dados em um banco de dados
document_ids = vector_store.add_documents(documents=all_splits)

In [9]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Criação do prompt injetando a persona exigida pela Sprint 2
prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        "Você é o ChargeGrid Intelligence, um assistente virtual especializado da GoodWe para soluções de recarga comercial de veículos elétricos (EV).\n"
        "Seu objetivo é responder às perguntas de forma clara e concisa usando o contexto fornecido abaixo. "
        "Se você não souber a resposta ou se a informação não estiver no contexto, diga educadamente que não possui essa informação técnica no momento.\n\n"
        "Contexto Técnico:\n{context}"
    ),
    MessagesPlaceholder(variable_name="messages"),
])

In [10]:
from langchain_core.documents import Document
from typing_extensions import List, TypedDict
from typing import Annotated, List
from typing_extensions import TypedDict
from langchain_core.documents import Document
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

# O novo estado agora rastreia o histórico de mensagens e os documentos recuperados
class State(TypedDict):
    messages: Annotated[list, add_messages]
    context: List[Document]

def retrieve(state: State):
    # A pergunta atual do usuário será sempre a última mensagem textual enviada no histórico
    ultima_mensagem = state["messages"][-1].content
    retrieved_docs = vector_store.similarity_search(ultima_mensagem)
    return {"context": retrieved_docs}

def generate(state: State):
    # Consolida o conteúdo dos documentos recuperados
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])

    # Alimenta o template com o contexto técnico e a lista completa de mensagens (histórico)
    prompt_formatado = prompt_template.invoke({
        "context": docs_content,
        "messages": state["messages"]
    })

    response = llm.invoke(prompt_formatado)

    # Retorna a resposta que será automaticamente anexada ao histórico de mensagens
    return {"messages": [response]}

#definindo o workflow
from langgraph.graph import START, StateGraph
from langgraph.checkpoint.memory import MemorySaver

# Monta o fluxo sequencial
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")

# O MemorySaver permite que o chatbot se lembre do contexto em chamadas consecutivas
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
from langchain_core.messages import HumanMessage

# Criamos uma thread para o teste isolado
config = {"configurable": {"thread_id": "teste_avulso_1"}}

# Invocamos passando o formato correto de mensagens
result = graph.invoke({"messages": [HumanMessage(content="A bateria do carro elétrico precisa estar completamente descarregada antes de carregá-lá?")]}, config)

print(f'Context: {result["context"]}\n\n')
# Buscamos o conteúdo da última mensagem gerada pela IA
print(f'Answer: {result["messages"][-1].content}')

In [ ]:
# Pergunta teste 02
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "teste_avulso_2"}}

result = graph.invoke({"messages": [HumanMessage(content="Quais são os tipos de carregadores de carro elétrico?")]}, config)

print(f'Context: {result["context"]}\n\n')
print(f'Answer: {result["messages"][-1].content}')

In [ ]:
# Pergunta teste 03
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "teste_avulso_3"}}

result = graph.invoke({"messages": [HumanMessage(content="Quanto custa uma recarga de um carro elétrico?")]}, config)

print(f'Context: {result["context"]}\n\n')
print(f'Answer: {result["messages"][-1].content}')

In [ ]:
# Pergunta teste 04
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "teste_avulso_4"}}

result = graph.invoke({"messages": [HumanMessage(content="Quanto tempo demora para realizar uma recarga de um carro elétrico?")]}, config)

print(f'Context: {result["context"]}\n\n')
print(f'Answer: {result["messages"][-1].content}')

In [ ]:
# Pergunta teste 05
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "teste_avulso_5"}}

result = graph.invoke({"messages": [HumanMessage(content="O carro elétrico pode ser recarregado debaixo de chuva?")]}, config)

print(f'Context: {result["context"]}\n\n')
print(f'Answer: {result["messages"][-1].content}')

In [ ]:
import uuid
from langchain_core.messages import HumanMessage

# Geramos uma sessão de chat contínua
session_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("⚡ Chatbot GoodWe (ChargeGrid Intelligence) Ativo!")
print("Converse com o assistente. Digite 'sair' para encerrar.\n")

while True:
    usuario_input = input("Você: ")
    if usuario_input.lower() in ["sair", "exit", "quit"]:
        print("Encerrando atendimento. Até logo!")
        break

    if not usuario_input.strip():
        continue

    # Executa o grafo passando a nova mensagem dentro da mesma thread_id (mantendo a memória ativa)
    resposta_grafo = graph.invoke({"messages": [HumanMessage(content=usuario_input)]}, session_config)

    # Pega a última mensagem retornada pelo nó de geração (que é a IA falando)
    print(f"\n🤖 Assistente:\n{resposta_grafo['messages'][-1].content}\n")
    print("="*40)